In [1]:
from pathlib import Path
ROOT = Path("..").resolve()
print("Project root:", ROOT)
for d in ["src/lbpscreen", "tests", "legacy", "data"]:
    print(f"{d:15s} exists: {(ROOT / d).exists()}")

Project root: C:\Users\IPRDC\lbp-screen
src/lbpscreen   exists: True
tests           exists: True
legacy          exists: True
data            exists: True


In [2]:
%%writefile ../pyproject.toml
[build-system]
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"

[project]
name = "lbpscreen"
version = "0.1.0"
description = "In silico screening of bacterial strains for live biotherapeutics using AGORA2 genome-scale metabolic models"
readme = "README.md"
requires-python = ">=3.9"
authors = [{ name = "Francis John Raymundo" }]
license = { text = "MIT" }
dependencies = ["cobra>=0.29", "pandas>=1.5", "scipy>=1.9"]

[project.optional-dependencies]
dev = ["pytest>=7", "jupyterlab"]

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]

Writing ../pyproject.toml


In [3]:
(ROOT / "README.md").write_text(
    "# lbpscreen\n\nIn silico screening of bacterial strains for live "
    "biotherapeutic products using AGORA2 genome-scale metabolic models.\n",
    encoding="utf-8")

(ROOT / ".gitignore").write_text("\n".join([
    "data/raw/", "data/processed/", "__pycache__/", "*.egg-info/",
    ".pytest_cache/", ".ipynb_checkpoints/", "*.pyc", ""]), encoding="utf-8")
print("done")

done


In [5]:
%%writefile ../src/lbpscreen/io.py
"""
lbpscreen.io - reading AGORA2 models in any of their distributed formats.

Author: Francis John Raymundo
"""

from pathlib import Path
import gzip
import re
import warnings

import cobra
import pandas as pd

__all__ = ["SUPPORTED_FORMATS", "detect_format", "read_model", "load_models",
           "model_summary", "sbml_gene_labels", "load_gene_labels"]

# File ending -> format name. Longer endings first so ".xml.gz" wins over ".gz".
SUPPORTED_FORMATS = {
    ".xml.gz": "sbml_gz",
    ".sbml.gz": "sbml_gz",
    ".xml": "sbml",
    ".sbml": "sbml",
    ".mat": "matlab",
    ".json": "json",
    ".yml": "yaml",
    ".yaml": "yaml",
}


def detect_format(path):
    """Return the format name for a model file, judged by its ending."""
    name = Path(path).name.lower()
    for ending, fmt in SUPPORTED_FORMATS.items():
        if name.endswith(ending):
            return fmt
    raise ValueError(
        f"Unrecognised model file {Path(path).name!r}. "
        f"Supported endings: {', '.join(SUPPORTED_FORMATS)}"
    )


def read_model(path, quiet=True):
    """Read ONE model file, whatever its format, into a cobra.Model.

    AGORA2 is distributed as SBML (.xml) and as MATLAB (.mat). SBML is
    the reference format for this package. A .mat model of the same
    strain uses different identifier spellings (checked on Day 2), so
    do not mix formats within one comparison without harmonising ids.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{path} does not exist")
    fmt = detect_format(path)

    with warnings.catch_warnings():
        if quiet:
            warnings.simplefilter("ignore")
        if fmt == "sbml":
            model = cobra.io.read_sbml_model(str(path))
        elif fmt == "sbml_gz":
            with gzip.open(path, "rt", encoding="utf-8") as handle:
                model = cobra.io.read_sbml_model(handle.read())
        elif fmt == "matlab":
            try:
                model = cobra.io.load_matlab_model(str(path))
            except NotImplementedError as err:
                # scipy cannot read MATLAB v7.3 (HDF5) files.
                raise ValueError(
                    f"{path.name} looks like a MATLAB v7.3 file, which "
                    "scipy cannot read. Use the SBML version instead."
                ) from err
        elif fmt == "json":
            model = cobra.io.load_json_model(str(path))
        else:  # yaml
            model = cobra.io.load_yaml_model(str(path))

    # Record where the model came from; useful in every later report.
    model.notes["source_file"] = path.name
    model.notes["source_format"] = fmt
    return model


def load_models(model_dir, file_map=None, quiet=True):
    """Load several models into a dict {label: cobra.Model}.

    file_map : {label: filename}. If None, every supported model file in
               model_dir is loaded and labelled by its file name.
    """
    model_dir = Path(model_dir)
    if file_map is None:
        file_map = {}
        for p in sorted(model_dir.iterdir()):
            try:
                detect_format(p)
            except ValueError:
                continue  # not a model file; skip it
            label = p.name.split(".")[0]
            if label in file_map:
                raise ValueError(
                    f"Two files share the label {label!r} (e.g. .xml and "
                    ".mat of one strain). Pass an explicit file_map."
                )
            file_map[label] = p.name
    return {label: read_model(model_dir / fname, quiet=quiet)
            for label, fname in file_map.items()}


def model_summary(models):
    """One row per model: sizes, objective and source format."""
    rows = []
    for label, m in models.items():
        obj = [r.id for r in m.reactions if r.objective_coefficient != 0]
        rows.append({
            "strain": label,
            "reactions": len(m.reactions),
            "metabolites": len(m.metabolites),
            "genes": len(m.genes),
            "exchanges": len(m.exchanges),
            "objective": obj[0] if obj else None,
            "format": m.notes.get("source_format"),
        })
    return pd.DataFrame(rows).set_index("strain")


def sbml_gene_labels(path):
    """Map {cobra_gene_id: original fbc:label} by reading the SBML text.

    SBML identifiers may not contain dots, so the file itself stores
    WP_002356251.1 as fbc:id="G_WP_002356251_1" and keeps the true name
    only in fbc:label. COBRApy keeps the id and drops the label. The
    dot -> underscore change is NOT reversible (EFMMH594_RS13230 already
    had an underscore), so the label must be read from the file.

    Attributes are found one at a time, so their order inside the tag
    does not matter.
    """
    path = Path(path)
    if detect_format(path) == "sbml_gz":
        with gzip.open(path, "rt", encoding="utf-8") as handle:
            text = handle.read()
    else:
        text = path.read_text(encoding="utf-8")

    labels = {}
    for tag in re.findall(r"<fbc:geneProduct\b[^>]*>", text):
        fid = re.search(r'\bfbc:id="([^"]+)"', tag)
        lab = re.search(r'\bfbc:label="([^"]*)"', tag)
        if fid and lab:
            key = fid.group(1)
            key = key[2:] if key.startswith("G_") else key
            labels[key] = lab.group(1)
    return labels


def load_gene_labels(model_dir, file_map):
    """Label maps for several SBML models: {strain: {gene_id: label}}.

    Takes the same arguments as load_models, so the two stay in step.
    Non-SBML files are skipped: .mat files keep the original gene names.
    """
    model_dir = Path(model_dir)
    out = {}
    for label, fname in file_map.items():
        if detect_format(model_dir / fname) in ("sbml", "sbml_gz"):
            out[label] = sbml_gene_labels(model_dir / fname)
    return out

Overwriting ../src/lbpscreen/io.py


In [7]:
import ast

OLD = ROOT / "legacy" / "gemtools.py"
PKG = ROOT / "src" / "lbpscreen"

MODULES = {   # module: (description, items to move, imports it needs)
    "search": ("Searching models for metabolites and reactions.",
               ["assert_present", "search_metabolites",
                "reactions_containing", "exchange_audit"],
               "import pandas as pd\n"),
    "gpr":    ("Gene-protein-reaction rules.",
               ["split_gpr", "classify_gpr", "gpr_summary"],
               "import re\n\nimport pandas as pd\n"),
    "media":  ("Diets and growth media.",
               ["vmh_to_cobra", "load_diet", "apply_medium", "diet_coverage"],
               "import pandas as pd\n"),
    "genes":  ("Decoding AGORA2 gene identifiers.",
               ["GENE_ID_PATTERNS", "classify_gene_id",
                "gene_id_census", "decode_bvbrc"],
               "import re\n\nimport pandas as pd\n"),
    "blast":  ("Parsing and filtering tabular BLAST output.",
               ["BLAST_COLUMNS", "read_blast", "filter_hits"],
               "import pandas as pd\n"),
}

source = OLD.read_text(encoding="utf-8")
tree = ast.parse(source)

blocks = {}   # name -> exact source text
for node in tree.body:
    if isinstance(node, ast.FunctionDef):
        blocks[node.name] = ast.get_source_segment(source, node)
    elif isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name):
        blocks[node.targets[0].id] = ast.get_source_segment(source, node)

for mod, (doc, names, imports) in MODULES.items():
    public = [n for n in names if not n.isupper() or n == "BLAST_COLUMNS"]
    text = (f'"""\nlbpscreen.{mod} - {doc}\n\nAuthor: Francis John Raymundo\n"""\n\n'
            f"{imports}\n__all__ = {public!r}\n\n\n"
            + "\n\n\n".join(blocks[n] for n in names) + "\n")
    (PKG / f"{mod}.py").write_text(text, encoding="utf-8")
    print(f"wrote {mod}.py  ({len(names)} items)")

wrote search.py  (4 items)
wrote gpr.py  (3 items)
wrote media.py  (4 items)
wrote genes.py  (4 items)
wrote blast.py  (3 items)


In [8]:
%%writefile ../src/lbpscreen/__init__.py
"""
lbpscreen - in silico screening of bacterial strains for live
biotherapeutic products (LBPs) using AGORA2 genome-scale metabolic models.

Author: Francis John Raymundo
"""

__version__ = "0.1.0"

from .io import *        # noqa: F401,F403
from .search import *    # noqa: F401,F403
from .gpr import *       # noqa: F401,F403
from .media import *     # noqa: F401,F403
from .genes import *     # noqa: F401,F403
from .blast import *     # noqa: F401,F403
from .genes import GENE_ID_PATTERNS  # noqa: F401

Writing ../src/lbpscreen/__init__.py


In [10]:
%%writefile ../tests/test_unit.py
"""Fast tests that need no model files. Run: pytest"""
import pandas as pd
import pytest

import lbpscreen as lbp


def test_detect_format():
    assert lbp.detect_format("a/b/Strain_AGORA2.xml") == "sbml"
    assert lbp.detect_format("Strain.XML.GZ") == "sbml_gz"
    assert lbp.detect_format("Strain.mat") == "matlab"
    with pytest.raises(ValueError):
        lbp.detect_format("notes.txt")


def test_vmh_to_cobra():
    assert lbp.vmh_to_cobra("EX_glc_D[e]") == "EX_glc_D_e_"


def test_gpr():
    assert lbp.split_gpr("a or b or c") == ["a", "b", "c"]
    assert lbp.classify_gpr("") == "none"
    assert lbp.classify_gpr("(a and b) or c") == "complex + isozymes"


@pytest.mark.parametrize("label, expected", [
    ("WP_002356251.1",      "refseq_protein"),
    ("349741.23.peg.853",   "bvbrc_peg_canonical"),
    ("1158653.5.620.peg",   "bvbrc_peg_feature"),
    ("479437.5.492_46.peg", "bvbrc_peg_reordered"),
    ("EFMMH594_RS13230",    "ncbi_locus_tag"),
])
def test_classify_gene_id(label, expected):
    assert lbp.classify_gene_id(label) == expected


def test_decode_bvbrc():
    d = lbp.decode_bvbrc("1158653.5.620.peg")
    assert d["fig_id"] == "fig|1158653.5.peg.620"
    assert d["ordering"] == "reordered"
    assert lbp.decode_bvbrc("EFMMH594_RS13230") is None


def test_sbml_gene_labels_any_attribute_order(tmp_path):
    xml = tmp_path / "toy.xml"
    xml.write_text(
        '<fbc:geneProduct fbc:id="G_WP_1_1" fbc:label="WP_1.1"/>\n'
        '<fbc:geneProduct fbc:label="X_RS9" fbc:id="G_X_RS9"/>\n',
        encoding="utf-8")
    assert lbp.sbml_gene_labels(xml) == {"WP_1_1": "WP_1.1", "X_RS9": "X_RS9"}


def test_filter_hits():
    fake = pd.DataFrame({
        "qseqid": ["real_full", "noise_short", "real_divergent"],
        "sseqid": ["x", "x", "x"],
        "pident": [99.9, 50.0, 58.1],
        "length": [1017, 12, 289],
        "mismatch": [1, 6, 121], "gapopen": [0, 0, 0],
        "qstart": [1, 51, 1], "qend": [1017, 62, 289],
        "sstart": [1, 487, 1], "send": [1017, 498, 289],
        "evalue": [0.0, 3.1, 2e-108],
        "bitscore": [2117.0, 16.2, 326.0],
        "qlen": [1017, 1017, 289], "slen": [1017, 1017, 289],
        "qcovs": [100, 1, 100],
    })
    res = lbp.filter_hits(fake)
    assert res["passes"].tolist() == [True, False, True]
    assert "coverage" in res.loc[1, "failed"]

Overwriting ../tests/test_unit.py


In [11]:
%%writefile ../tests/test_models.py
"""Tests on the real AGORA2 models. Skipped if the files are not present."""
from pathlib import Path

import pytest

import lbpscreen as lbp

ROOT = Path(__file__).resolve().parent.parent
MODEL_DIR = ROOT / "data/processed/core_models"
DIETS = ROOT / "data/raw/diets"

FILE_MAP = {
    "E_faecalis_MMH594":    "Enterococcus_faecalis_MMH594_AGORA2_v2.01.xml",
    "E_lenta_DSM2243":      "Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml",
    "E_coli_Nissle1917":    "Escherichia_coli_Nissle_1917_AGORA2_v2.01.xml",
    "A_muciniphila_BAA835": "Akkermansia_muciniphila_ATCC_BAA_835_AGORA2_v2.01.xml",
}

needs_models = pytest.mark.skipif(
    not all((MODEL_DIR / f).exists() for f in FILE_MAP.values()),
    reason="AGORA2 model files not found in data/processed/core_models",
)


@pytest.fixture(scope="module")
def models():
    return lbp.load_models(MODEL_DIR, FILE_MAP)


@needs_models
def test_sizes(models):
    assert len(models) == 4
    assert len(models["E_faecalis_MMH594"].reactions) == 1558
    assert len(models["E_lenta_DSM2243"].reactions) == 1035


@needs_models
def test_positive_and_negative_controls(models):
    lbp.assert_present(models, "levdopa_c", "E_faecalis_MMH594")
    lbp.assert_present(models, "mtym_c", "E_lenta_DSM2243")
    with pytest.raises(AssertionError):
        lbp.assert_present(models, "levodopa_c", "E_faecalis_MMH594")  # typo


@needs_models
def test_audit(models):
    audit = lbp.reactions_containing(models, ["levdopa", "dpmn", "tym_", "mtym"])
    assert len(audit) == 13


@needs_models
def test_gene_labels():
    ef = lbp.load_gene_labels(MODEL_DIR, FILE_MAP)["E_faecalis_MMH594"]
    assert len(ef) == 1140
    assert ef["1158653_5_620_peg"] == "1158653.5.620.peg"
    assert ef["WP_002356251_1"] == "WP_002356251.1"


@pytest.mark.skipif(not (DIETS / "high-fiber.csv").exists(),
                    reason="VMH diet file not found")
def test_diet():
    diet = lbp.load_diet(DIETS / "high-fiber.csv")
    assert len(diet) == 91
    assert abs(diet["EX_glc_D_e_"] - 120.211) < 1e-3

Writing ../tests/test_models.py
